In [6]:
import sys
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader
sys.path.append(os.path.abspath(os.path.join(os.pardir, "src")))
from data_loader import image_downloader
from image_lenet import ListingsDataset
from image_lenet import LeNet5

In [7]:
#load the dataset and save image urls and listings ids in a dictionary
#DONT RUN THIS UNLESS YOU WANT TO DOWNLOAD ALL IMAGES IT TAKES A LONG TIME
df = pd.read_csv("../data/listings.csv")
id_image_dicct = {}
for index, row in df.iterrows():
    if pd.notna(row["picture_url"]):
        id_image_dicct[row["id"]] = row["picture_url"]
#download images
#for listing_id, image_url in id_image_dicct.items():
#    image_downloader(image_url, listing_id, True)

In [8]:
# set up transforms and dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    # Add normalization if needed
])

dataset = ListingsDataset(csv_file="../data/listings.csv", img_dir="../data/image_per_listing", transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [9]:
# Define device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [10]:
model = LeNet5().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, prices in dataloader:
        images = images.to(device)
        prices = prices.float().unsqueeze(1).to(device)  # Ensure shape [batch, 1]

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, prices)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/10, Loss: 184477.3968
Epoch 2/10, Loss: 179761.2145
Epoch 3/10, Loss: 180455.7786
Epoch 4/10, Loss: 179703.5002
Epoch 5/10, Loss: 179708.1432
Epoch 6/10, Loss: 179392.3096
Epoch 7/10, Loss: 179329.6409
Epoch 8/10, Loss: 179210.6873
Epoch 9/10, Loss: 179376.7117
Epoch 10/10, Loss: 179104.9488


In [15]:
# Predict the price of a single image using the trained model
model.eval()
with torch.no_grad():
    image = images[12].unsqueeze(0).to(device)  # Select one image and add batch dimension
    predicted_price = model(image)
    print(f"Predicted price: {predicted_price.item():.2f}")

Predicted price: 147.16
